# PCA workflow for clustering analysis

Notebook organizado para uso em repositório GitHub.

## Observações
- Os **dados não estão incluídos** neste repositório.
- Ajuste os caminhos na seção **Configuração inicial** antes de executar.
- A estrutura sugerida é:

```text
data/
├── raw/
│   └── 3_MEDIA_DO_BANCO_DE_DADOS-COMPLETO-2026.xlsx
├── processed/
│   └── BANCO_SEPARADO_POR_CLUSTER.xlsx
└── outputs/
```


In [ ]:
# Instale as dependências, se necessário
# %pip install pca factor_analyzer

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# Configuração inicial
# ==========================================
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "outputs"

ARQUIVO_BASE = RAW_DIR / "3_MEDIA_DO_BANCO_DE_DADOS-COMPLETO-2026.xlsx"
ARQUIVO_CLUSTERS = PROCESSED_DIR / "BANCO_SEPARADO_POR_CLUSTER.xlsx"

print(f"Arquivo principal: {ARQUIVO_BASE}")
print(f"Arquivo de clusters: {ARQUIVO_CLUSTERS}")

In [ ]:
dados = pd.read_excel(ARQUIVO_BASE)
dados.head()

## 1. Inspeção inicial dos dados

In [ ]:
dados.info()

In [ ]:
# Verificar se existe algum NaN no DataFrame
tem_nan = dados.isnull().values.any()
print(f"O DataFrame contém NaNs? {tem_nan}")

In [ ]:
# Contar o número total de NaNs por coluna
print("\nContagem de NaNs por coluna:")
print(dados.isnull().sum())

In [ ]:
# 2. Encontrar colunas que contêm pelo menos um valor NaN
print("Colunas com valores NaN:")
# dados.isnull() ou dados.isna() retorna um DataFrame booleano (True para NaN)
# .any() verifica se há algum True em cada coluna
# O resultado é uma Série booleana onde o índice é o nome da coluna
print(dados.isnull().any())
print("-" * 30)

# 3. Contar o número total de NaNs por coluna
print("Contagem de NaNs por coluna:")
# .sum() soma os valores True (que são tratados como 1) e False (como 0)
print(dados.isnull().sum())
print("-" * 30)

# 4. Listar as linhas e colunas exatas onde os NaNs estão localizados
print("Localização exata dos NaNs (linhas e colunas):")
# dados.stack() transforma o DataFrame em uma Série com multi-índice (linha, coluna)
# isnull() verifica os NaNs na Série
# [dados.isnull().stack()] filtra apenas os valores True (os NaNs)
nan_locations = dados.isnull().stack()[dados.isnull().stack()]
print(nan_locations)
# Para uma visualização mais limpa do índice e da coluna:
print("\nÍndice (linha) e Nome da Coluna:")
for index, value in nan_locations.items():
    print(f"NaN encontrado na linha/índice: {index[0]}, Coluna: {index[1]}")

## PCA - Principal componente analysis

## 2. Preparação para PCA

In [ ]:
# para a análise de PCA temos que ter um indice
dados = dados.set_index('Municipio')
dados.head()

In [ ]:
# Preprocessing: Convert object columns to numeric, handling comma decimals
for col in dados.columns:
    if dados[col].dtype == 'object':
        # Replace comma with period before converting to numeric
        dados[col] = dados[col].astype(str).str.replace(',', '.', regex=False)
        # Attempt conversion, errors='coerce' will turn values that cannot be parsed into NaN.
        dados[col] = pd.to_numeric(dados[col], errors='coerce')

# estilo Jedi
plt.figure(figsize=(10,5))
contador =0
# Collect numeric columns for plotting. The original subplot setup (4,2) allows for 8 plots.
numeric_columns_to_plot = [col for col in dados.columns[1:] if pd.api.types.is_numeric_dtype(dados[col])]

for i in numeric_columns_to_plot:
    if contador >= 8: # Limit to 8 plots to fit the 4x2 subplot layout
        break
    contador +=1 # contador = contador +1
    plt.subplot(4,2,contador)
    sns.boxplot(y=i, x=dados.index, data=dados,
                showmeans=True,
                meanprops={'marker':'o',
                'markerfacecolor':'white',
                'markeredgecolor':'black',
                'markersize':'6'})
    plt.xlabel(' ')
    if contador in [7, 8]: # para os gráficos 7 e 8
        plt.xticks(rotation=90) # rotaciona as legendas
    else:
        plt.xticks([]) # remove as legendas do eixo x para todos os outros gráficos

plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show() # Display the plot

### Padronizar os dados

In [ ]:
# imprescindível padronizar os dados
from scipy import stats

var_selecionadas = dados.loc[:,'Formação_Florestal':]
dados_pad = pd.DataFrame(stats.zscore(var_selecionadas), columns = var_selecionadas.columns) # todas as features agora tem unidades em desvios padrão
dados_pad.head(3)

## 3. Testes de adequação da PCA

### 2 Testes essenciais para verificar se podemos utilizar a PCA e AF
**Teste de (esfericidade) Bartlett:**<br>
verifica se existe correlação suficiente entre as variáveis para justificar a redução dimensional (existe uma estrutura interna)<br>
ok se valor <0.05 <br>
**Critério (ou Teste) de Kaiser-Meyer-Olkin (KMO)**<br>
verifica se existe variância comum entre as variáveis (padronizadas)<br>
ok se valor >0.05

In [ ]:
# %pip install factor_analyzer

In [ ]:
import numpy as np
import pandas as pd

# variância por coluna
variancias = dados_pad.var(axis=0)

colunas_constantes = variancias[variancias == 0].index.tolist()

print("Colunas com variância zero:")
print(colunas_constantes)


In [ ]:
# matriz de correlação
corr = pd.DataFrame(dados_pad).corr().abs()

# máscara da diagonal
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# colunas com correlação perfeita
colunas_perfeitas = [
    col for col in upper.columns if any(upper[col] >= 0.9999)
]

print("Colunas perfeitamente correlacionadas:")
print(colunas_perfeitas)


In [ ]:
import numpy as np
import pandas as pd

X = pd.DataFrame(dados_pad).copy()

print("Shape inicial:", X.shape)

# 1) Checar NaN / inf
finite_mask = np.isfinite(X.to_numpy())
cols_with_nonfinite = X.columns[~finite_mask.all(axis=0)].tolist()
rows_with_nonfinite = np.where(~finite_mask.all(axis=1))[0].tolist()

print("\nColunas com NaN/inf:", cols_with_nonfinite[:20], "..." if len(cols_with_nonfinite) > 20 else "")
print("Qtd colunas com NaN/inf:", len(cols_with_nonfinite))
print("Qtd linhas com NaN/inf:", len(rows_with_nonfinite))

# 2) Remover colunas all-NaN e linhas com qualquer NaN/inf
X1 = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all")
X1 = X1.dropna(axis=0, how="any")
print("\nApós remover NaN/inf (linhas/colunas):", X1.shape)

# 3) Remover colunas com variância ~0 (zscore pode gerar NaN e mascarar isso)
var = X1.var(axis=0)
zero_like = var[var <= 1e-12].index.tolist()
print("\nColunas com variância ~0:", zero_like)

X2 = X1.drop(columns=zero_like) if zero_like else X1
print("Após remover variância ~0:", X2.shape)

# 4) Checar correlação quase perfeita (redundância par-a-par)
corr = X2.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if (upper[col] >= 0.9999).any()]

print("\nColunas com |corr| >= 0.9999 (candidatas a drop):", to_drop_corr[:30], "..." if len(to_drop_corr) > 30 else "")
print("Qtd:", len(to_drop_corr))

X3 = X2.drop(columns=to_drop_corr) if to_drop_corr else X2
print("Após remover correlação ~perfeita:", X3.shape)

# 5) Rank numérico (se rank < p, Bartlett dá problema)
A = X3.to_numpy(dtype=float)
u, s, vt = np.linalg.svd(A, full_matrices=False)
tol = s.max() * max(A.shape) * np.finfo(float).eps
rank = int((s > tol).sum())

print(f"\nRank numérico: {rank} | Nº colunas (p): {A.shape[1]}")
print("Menores singular values:", s[-min(10, len(s)) :])

# Resultado final para usar
dados_bartlett = X3
print("\n✅ Use 'dados_bartlett' no Bartlett e no PCA. Shape:", dados_bartlett.shape)

In [ ]:
import numpy as np
import pandas as pd
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

def preparar_para_bartlett(X: pd.DataFrame,
                          var_tol: float = 1e-12,
                          corr_pair_tol: float = 0.9999,
                          eig_tol: float = 1e-10,
                          verbose: bool = True):
    X = pd.DataFrame(X).copy()

    # 1) troca inf por NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # 2) remove colunas all-NaN e linhas com qualquer NaN
    X = X.dropna(axis=1, how="all")
    X = X.dropna(axis=0, how="any")

    # 3) remove colunas com variância ~0
    var = X.var(axis=0)
    drop_var = var[var <= var_tol].index.tolist()
    if drop_var:
        X = X.drop(columns=drop_var)

    # 4) remove pares com correlação quase perfeita (redundância 1:1)
    if X.shape[1] >= 2:
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop_corr = [c for c in upper.columns if (upper[c] >= corr_pair_tol).any()]
        if drop_corr:
            X = X.drop(columns=drop_corr)
    else:
        drop_corr = []

    # 5) se ainda estiver singular “em conjunto”, remove colunas guiado por menor autovalor
    dropped_eig = []
    while X.shape[1] >= 2:
        C = X.corr().to_numpy()
        eigvals, eigvecs = np.linalg.eigh(C)  # autovalores em ordem crescente
        if eigvals[0] > eig_tol:
            break  # ok, correlação não-singular

        # pega a direção do menor autovalor e remove a coluna com maior contribuição absoluta
        v = eigvecs[:, 0]
        idx_remove = int(np.argmax(np.abs(v)))
        col_remove = X.columns[idx_remove]
        dropped_eig.append(col_remove)
        X = X.drop(columns=[col_remove])

    # checagens finais
    corr_det = np.linalg.det(X.corr().to_numpy()) if X.shape[1] >= 2 else np.nan

    if verbose:
        print("Shape final para Bartlett:", X.shape)
        print("Removidas (variância ~0):", drop_var)
        print("Removidas (corr ~perfeita):", drop_corr)
        print("Removidas (singularidade via autovalor):", dropped_eig)
        print("det(corr) final:", corr_det)

    return X

# ✅ USE ISSO no lugar do seu Bartlett atual
dados_bartlett = preparar_para_bartlett(dados_pad, verbose=True)

x2, pvalor = calculate_bartlett_sphericity(dados_bartlett)
x2, pvalor

In [ ]:
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

x2, pvalor = calculate_bartlett_sphericity(dados_bartlett)
x2, pvalor


In [ ]:
# Barlett
#from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
#x2, pvalor = calculate_bartlett_sphericity(dados_pad)
#x2, pvalor

# 2° valor <0.05 OK temos homocedasticidade das features
# PCA liberado

In [ ]:
# KMO
from factor_analyzer.factor_analyzer import calculate_kmo
import numpy as np

# Handle NaN values and zero-variance columns in dados_pad before calculating KMO
# Create a cleaned copy to avoid modifying the original dados_pad unexpectedly
dados_pad_cleaned = dados_pad.copy()

# 1. Drop columns that are entirely NaN
dados_pad_cleaned = dados_pad_cleaned.dropna(axis=1, how='all')

# 2. Drop rows that still contain any NaN values
dados_pad_cleaned = dados_pad_cleaned.dropna(axis=0)

# 3. Check for and remove columns with zero variance, as they cause issues for correlation/SVD
# Use .values.var() to ensure correct variance calculation for pandas Series/DataFrames
zero_variance_cols = dados_pad_cleaned.columns[dados_pad_cleaned.var() == 0]
if not zero_variance_cols.empty:
    print(f"Warning: Columns with zero variance found and removed: {list(zero_variance_cols)}")
    dados_pad_cleaned = dados_pad_cleaned.drop(columns=zero_variance_cols)

# Ensure the DataFrame is not empty after cleaning
if dados_pad_cleaned.empty:
    print("DataFrame is empty after removing NaN and zero-variance columns. Cannot perform KMO test.")
else:
    kmo, kmo_model = calculate_kmo(dados_pad_cleaned)
    print(f"KMO value: {kmo}")
    # KMO model can be very large if many features, print head or summary if needed
    # For now, just print the kmo_model result which is an array of KMO for each variable
    # print(f"KMO model values:\n{kmo_model}") # Uncomment if you want to see all individual KMO values

#(0<= kmo <=1) --> Índice
# se KMO >0.05 OK podemos realizar a PCA
# autores ideal seria >0.7
# 0.6 - limite do medíocre


In [ ]:
from pca import pca # Ensure the pca library is imported

modelo = pca(n_components = 0.95) # faça quantos pcs nescessarios para explicar 95% dos dados
# Use the cleaned DataFrame for PCA
resultado= modelo.fit_transform(dados_pad_cleaned)

In [ ]:
resultado

## 4. Ajuste do modelo PCA e visualizações iniciais

### Gráfico PCA

In [ ]:
modelo.scatter(density=True, SPE=True, legend=True, label='Scores PCA')
plt.plot([0,0],[-7,7],'k-') #inserindo o eixo Y
plt.plot([-7,7],[0,0],'k-'); #inserindo o eixo X
#plt.xlim(-7,7)
#plt.ylim(-7,7)

In [ ]:
modelo.biplot(#SPE=True,
            cmap='Set1',
            legend=1)
plt.plot([0,0],[-3,3],'k-') #inserindo o eixo Y
plt.plot([-5,5],[0,0],'k-'); #inserindo o eixo X
#plt.xlim(-5,5)
#plt.ylim(-3,3)

In [ ]:
loadings = modelo.results['loadings']

# Criar Series para PC1 e PC2
loadings_pc1 = pd.Series(loadings.loc['PC1'])
loadings_pc2 = pd.Series(loadings.loc['PC2'])

top_pc1 = loadings_pc1.abs().sort_values(ascending=False).head()
print(top_pc1)

top_pc2 = loadings_pc2.abs().sort_values(ascending=False).head()
print(top_pc2)

In [ ]:
modelo.scatter(density=True, SPE=True)
plt.plot([0,0],[-7,7],'k-') #inserindo o eixo Y
plt.plot([-7,7],[0,0],'k-'); #inserindo o eixo X
#plt.xlim(-7,7)
#plt.ylim(-7,7)
modelo.biplot(#SPE=True,
            cmap='Set1',
            legend=1)
plt.plot([0,0],[-3,3],'k-') #inserindo o eixo Y
plt.plot([-5,5],[0,0],'k-'); #inserindo o eixo X
#plt.xlim(-5,5)
#plt.ylim(-3,3)

In [ ]:
modelo.check_verbosity()

In [ ]:
modelo.plot(figsize=(5,5))

In [ ]:
resultado['explained_var']

In [ ]:
resultado['outliers']

In [ ]:
resultado['topfeat'] #feature mais importante para cada componente

In [ ]:
# Gráfico 3D
from mpl_toolkits.mplot3d import Axes3D
modelo.biplot3d()

## 5. Ranking de variáveis

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# X deve ser o dataframe já padronizado e LIMPO (sem NaN/inf)
X = pd.DataFrame(dados_pad_cleaned)

# PCA completo
pca = PCA()
pca.fit(X)

# Loadings (pesos das variáveis)
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# Variância explicada
explained_var = pca.explained_variance_ratio_

# Importância global da feature (soma ponderada)
feature_importance = (
    loadings.abs()
    .mul(explained_var, axis=1)
    .sum(axis=1)
)

# Ranking
ranking = (
    feature_importance
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "feature", 0: "importance"})
)

# TOP 50
top_50 = ranking.head(50)
print(top_50)

## 6. Ajustes de visualização e biplots

# Novas alterações para visualização

In [ ]:
# =========================
# MANUAL (ultra-curto)
# =========================
MANUAL = {
    # IDs gerais
    "Municipio": "MU",
    "Área do municipio": "AR",
    "PMVA": "PV",
    "IDEB": "ID",

    # Land use / cover (mínimo possível com baixa ambiguidade)
    "Formação_Florestal": "FF",
    "Formação_Savânica": "FS",
    "Mangue": "MG",
    "Restinga_Arbórea": "RA",
    "Campo_Alagado_e_Área_Pantanosa": "WT",
    "Formação_Campestre": "FC",
    "Apicum": "AP",
    "Afloramento_Rochoso": "RR",
    "Restinga_Herbácea": "RH",
    "Praia_Duna_e_Areal": "SD",
    "Rio_Lago_e_Oceano": "WA",
    "Pastagem": "PA",
    "Soja": "SJ",
    "Cana": "CA",
    "Algodão": "AL",
    "Outras_Lavouras_Temporárias": "AT",  # Annual (other)
    "Café": "CF",
    "Citrus": "CT",
    "Outras_Lavouras_Perenes": "OP",      # Other perennial
    "Silvicultura": "SP",                 # Silvic. plantations
    "Mosaico_de_Usos": "MO",
    "Área_Urbanizada": "UR",
    "Mineração": "MN",
    "Aquicultura": "AQ",
    "Outras_Áreas_não_Vegetadas": "NV",

    # Demography
    "populacao": "P",
    "homens": "M",
    "mulheres": "F",
    "razao_sexo": "SR",
    "id_media": "AG",
    "dens_demog": "DN",

    # Education
    "Ensino_fundamental": "EF",
    "Ensino_infantil": "EI",
    "Ensino_médio": "EM",

    # Economy
    "PIB_per_capita": "GD",
    "Impostos": "TX",
    "VA_Agropecuária": "GA",
    "VA_Indústria": "GI",
    "VA_Serviços": "GS",
    "VA_Administração_pública": "GP",

    # Transport
    "VeicPasseio": "CR",
    "Coletivos": "BS",
    "VeicCarga": "TK",
    "DuasRodas": "2W",
    "TracReb": "TR",
}


# =========================
# 2) SUF_MAP (inglês)
# =========================
SUF_MAP = {
    "Verao": "S",     # Summer
    "Outono": "A",    # Autumn
    "Inverno": "W",   # Winter
    "Primavera": "P", # Spring
    "Total": "T",
    "Media": "M",
    "Vmax": "X",
    "Vmin": "N"
}


import re

# =========================
# 3) Função de abreviação (mais agressiva)
#    - remove ruídos (_SFC_)
#    - encurta NASA/clima
#    - fallback por iniciais (2-3 letras)
# =========================
def abbrev_var(var: str) -> str:
    if var in MANUAL:
        return MANUAL[var]

    tokens = re.split(r"[_\s]+", var.strip())

    # Extrai sufixos (estatística + estação)
    sufs = []
    while tokens and tokens[-1] in SUF_MAP:
        sufs.append(SUF_MAP[tokens.pop()])
    sufs = list(reversed(sufs))

    base = "_".join(tokens)

    # Padronização NASA / clima (bem curta)
    repl = {
        "ALLSKY": "AS",
        "CLRSKY": "CS",
        "PRECTOTCORR": "PPT",
        "CLOUD_AMT": "CLD",
        "GWETROOT": "GWRT",
        "GWETTOP": "GWTP",
        "SFC": "",          # remove ruído
        "DWN": "DN",
        "DNI": "DI",
        "DIFF": "DF",
        "PAR": "PAR",
        "SW": "SW",
        "LW": "LW",
        "UV_INDEX": "UVI",
        "UV": "UV",
        "PS": "PS",
        "TS": "TS",
        "QV2M": "QV2",
        "RH2M": "RH2",
        "T2MDEW": "TD2",
        "T2M_MAX": "T2X",
        "T2M_MIN": "T2N",
        "T2M": "T2",
        "WD10M": "WD",
        "WS10M_MAX": "W10X",
        "WS10M": "W10",
        "WS2M": "W2",
    }

    for k, v in repl.items():
        base = base.replace(k, v)

    base = re.sub(r"_+", "_", base).strip("_")

    # Se ainda ficar grande, usa iniciais (mantém números)
    if len(base) > 12:
        parts = base.split("_")
        new = []
        for p in parts:
            if re.search(r"\d", p):         # mantém tokens com números
                new.append(p)
            else:
                new.append(p[:2].upper())   # mais curto: 2 letras
        base = "_".join(new)

    return f"{base}_{'_'.join(sufs)}" if sufs else base


In [ ]:
variaveis = dados_pad_cleaned.columns.tolist()
siglas = {v: abbrev_var(v) for v in variaveis}

# inspeção rápida
for k in list(siglas)[:25]:
    print(f"{k:35s} -> {siglas[k]}")

In [ ]:
# =========================
# ✅ BIPLOT (pca lib) com siglas
# =========================
import pandas as pd
import matplotlib.pyplot as plt

# 1) Monte o "mapa" final de nomes -> siglas
#    (se siglas já é dict {nome_original: sigla}, perfeito)
#    garante fallback caso falte alguma chave
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

# 2) APLICA as siglas onde o biplot realmente lê: modelo.results['loadings'] (colunas!)
if "loadings" in modelo.results:
    modelo.results["loadings"] = modelo.results["loadings"].rename(columns=siglas_map)

# (opcional) também renomeia topfeat, caso você use tabelas/plots dela
if "topfeat" in modelo.results and isinstance(modelo.results["topfeat"], pd.DataFrame):
    if "feature" in modelo.results["topfeat"].columns:
        modelo.results["topfeat"]["feature"] = modelo.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

# 3) Agora sim: biplot vai sair com as siglas
modelo.biplot(cmap="Set1", legend=1)

# seus eixos (mantive igual ao seu padrão)
plt.plot([0, 0], [-3, 3], "k-")   # eixo Y
plt.plot([-5, 5], [0, 0], "k-")   # eixo X

plt.tight_layout()
plt.show()

# =========================
# 💾 SALVAR BIPLOT ATUAL
# =========================
import matplotlib.pyplot as plt

fig = plt.gcf()  # captura a figura do biplot que já está na tela

fig.savefig(
    "biplot_teste.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print("Imagem salva com sucesso: biplot_teste.png")

In [ ]:
# =========================
# ✅ BIPLOT (pca lib) COM SIGLAS, CORES EXATAS E SETAS LIMPAS
# =========================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 1) Carregar os clusters do arquivo indicado
# (caminho relativo para uso no repositório)
df_clusters = pd.read_excel(ARQUIVO_CLUSTERS)
clusters = df_clusters['cluster'].values

# 2) Monte o "mapa" final de nomes -> siglas
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

# 3) Aplica as siglas no modelo PCA
if "loadings" in modelo.results:
    modelo.results["loadings"] = modelo.results["loadings"].rename(columns=siglas_map)

if "topfeat" in modelo.results and isinstance(modelo.results["topfeat"], pd.DataFrame):
    if "feature" in modelo.results["topfeat"].columns:
        modelo.results["topfeat"]["feature"] = modelo.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

# 4) Dicionário de cores dos clusters (preenchimento e borda)
cluster_style = {
    0: {"fill": "#f5d899", "edge": "#e69f00"}, 1: {"fill": "#bbe1f6", "edge": "#56b4e9"},
    2: {"fill": "#99d8c7", "edge": "#009e73"}, 3: {"fill": "#f9f4b3", "edge": "#f0e442"},
    4: {"fill": "#99c7e0", "edge": "#0072b2"}, 5: {"fill": "#eebf99", "edge": "#d55e00"},
    6: {"fill": "#ebc9dc", "edge": "#cc79a7"}, 7: {"fill": "#bbac9b", "edge": "#543005"},
    8: {"fill": "#99b6b6", "edge": "#004949"}, 9: {"fill": "#d6d6d6", "edge": "#9a9a9a"}
}

# 5) Chama o biplot nativo com escala correta, mas com estética limpa
# - alpha=0.0: Oculta os pontos originais da biblioteca (vamos plotar os nossos por cima)
# - n_feat=20: Limita as setas para as 20 variáveis mais importantes, evitando poluição
modelo.biplot(
    n_feat=20, 
    legend=False, 
    show=False, 
    alpha=0.0, 
    fontdict={'color': 'black', 'size': 9, 'weight': 'bold'},
    arrowdict={'color': '#444444', 'linewidth': 0.8, 'alpha': 0.8}
)

# Captura a figura e eixos gerados pela biblioteca para customização
fig = plt.gcf()
ax = plt.gca()

# 6) Adiciona os pontos dos municípios coloridos pelos clusters
# 'PC' contém as coordenadas exatas e já escalonadas geradas pelo modelo
pcs = modelo.results['PC']  

for cluster_id, config in cluster_style.items():
    idx = (clusters == cluster_id)
    ax.scatter(
        pcs.iloc[idx, 0],  # Eixo PC1
        pcs.iloc[idx, 1],  # Eixo PC2
        c=config["fill"],
        edgecolors=config["edge"],
        label=f"Cluster {cluster_id}",
        s=60, 
        alpha=0.85, 
        linewidth=1.0,
        zorder=3
    )

# 7) Eixos centrais de referência
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, zorder=1)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, zorder=1)

# 8) Legenda customizada externa
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=f'Cluster {k}',
           markerfacecolor=v["fill"], markeredgecolor=v["edge"], markersize=10)
    for k, v in cluster_style.items() if k in clusters
]
ax.legend(handles=legend_elements, title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title("Biplot PCA - Variáveis e Clusters", pad=15)
plt.tight_layout()
plt.show()

# =========================
# 💾 SALVAR BIPLOT ATUAL
# =========================
save_path = "biplot_teste2.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"Imagem salva com sucesso: {save_path}")

In [ ]:
# =========================
# ✅ BIPLOT (pca lib) COM SIGLAS E CORES EXATAS (CORREÇÃO DE VISIBILIDADE)
# =========================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 1) Carregar os clusters do arquivo indicado
# (caminho relativo para uso no repositório)
df_clusters = pd.read_excel(ARQUIVO_CLUSTERS)
clusters = df_clusters['cluster'].values

# 2) Monte o "mapa" final de nomes -> siglas
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

# 3) Aplica as siglas no modelo PCA
if "loadings" in modelo.results:
    modelo.results["loadings"] = modelo.results["loadings"].rename(columns=siglas_map)

if "topfeat" in modelo.results and isinstance(modelo.results["topfeat"], pd.DataFrame):
    if "feature" in modelo.results["topfeat"].columns:
        modelo.results["topfeat"]["feature"] = modelo.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

# 4) Dicionário de cores dos clusters (preenchimento e borda)
cluster_style = {
    0: {"fill": "#f5d899", "edge": "#e69f00"}, 1: {"fill": "#bbe1f6", "edge": "#56b4e9"},
    2: {"fill": "#99d8c7", "edge": "#009e73"}, 3: {"fill": "#f9f4b3", "edge": "#f0e442"},
    4: {"fill": "#99c7e0", "edge": "#0072b2"}, 5: {"fill": "#eebf99", "edge": "#d55e00"},
    6: {"fill": "#ebc9dc", "edge": "#cc79a7"}, 7: {"fill": "#bbac9b", "edge": "#543005"},
    8: {"fill": "#99b6b6", "edge": "#004949"}, 9: {"fill": "#d6d6d6", "edge": "#9a9a9a"}
}

# 5) Chama o biplot nativo
modelo.biplot(legend=False)

fig = plt.gcf()
ax = plt.gca()

# 🛠️ TRUQUE PRAGMÁTICO: Oculta os pontos padrão usando set_visible em vez de set_alpha
for collection in ax.collections:
    collection.set_visible(False)

# 6) Adiciona os pontos dos municípios coloridos pelos clusters por cima
pcs = modelo.results['PC']  

for cluster_id, config in cluster_style.items():
    idx = (clusters == cluster_id)
    if idx.sum() > 0:
        ax.scatter(
            pcs.iloc[idx, 0],  # Eixo PC1
            pcs.iloc[idx, 1],  # Eixo PC2
            c=config["fill"],
            edgecolors=config["edge"],
            label=f"Cluster {cluster_id}",
            s=70, 
            alpha=0.9, 
            linewidth=1.0,
            zorder=3
        )

# 7) Eixos centrais de referência (seu padrão)
plt.plot([0, 0], [-3, 3], "k-", zorder=1)   # eixo Y
plt.plot([-5, 5], [0, 0], "k-", zorder=1)   # eixo X

# 8) Legenda customizada externa
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=f'Cluster {k}',
           markerfacecolor=v["fill"], markeredgecolor=v["edge"], markersize=10)
    for k, v in cluster_style.items() if k in set(clusters)
]
ax.legend(handles=legend_elements, title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title("Biplot PCA - Variáveis e Clusters", pad=15)
plt.tight_layout()

# =========================
# 💾 SALVAR BIPLOT ATUAL
# =========================
save_path = "biplot_teste3.png"

fig.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Imagem salva com sucesso: {save_path}")

plt.show()

In [ ]:
# =========================
# ✅ BIPLOT PADRÃO PUBLICAÇÃO (Environmental Science & Policy / MapBiomas)
# =========================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 1) Carregar os clusters do arquivo indicado
# (caminho relativo para uso no repositório)
df_clusters = pd.read_excel(ARQUIVO_CLUSTERS)
clusters = df_clusters['cluster'].values

# 2) Nomes e Siglas
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

if "loadings" in modelo.results:
    modelo.results["loadings"] = modelo.results["loadings"].rename(columns=siglas_map)
if "topfeat" in modelo.results and isinstance(modelo.results["topfeat"], pd.DataFrame):
    if "feature" in modelo.results["topfeat"].columns:
        modelo.results["topfeat"]["feature"] = modelo.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

# 3) Dicionário de Cores e NOMES GEOGRÁFICOS (Fundamental para o MapBiomas)
cluster_style = {
    0: {"fill": "#f5d899", "edge": "#e69f00", "name": "East-Central"}, 
    1: {"fill": "#bbe1f6", "edge": "#56b4e9", "name": "Northeast"},
    2: {"fill": "#99d8c7", "edge": "#009e73", "name": "Southern Coast"}, 
    3: {"fill": "#f9f4b3", "edge": "#f0e442", "name": "North"},
    4: {"fill": "#99c7e0", "edge": "#0072b2", "name": "Southwest"}, 
    5: {"fill": "#eebf99", "edge": "#d55e00", "name": "East"},
    6: {"fill": "#ebc9dc", "edge": "#cc79a7", "name": "West"}, 
    7: {"fill": "#bbac9b", "edge": "#543005", "name": "South"},
    8: {"fill": "#99b6b6", "edge": "#004949", "name": "Central"}, 
    9: {"fill": "#d6d6d6", "edge": "#9a9a9a", "name": "State Capital"}
}

# 4) Chama o biplot nativo
# Reduzido n_feat para 12 para eliminar a sobreposição ilegível de siglas
modelo.biplot(n_feat=18, legend=False) # <--------------------- numero de setas

fig = plt.gcf()
ax = plt.gca()

# Oculta os pontos padrão
for collection in ax.collections:
    collection.set_visible(False)

# 5) Adiciona os pontos dos municípios
pcs = modelo.results['PC']  
for cluster_id, config in cluster_style.items():
    idx = (clusters == cluster_id)
    if idx.sum() > 0:
        ax.scatter(
            pcs.iloc[idx, 0], pcs.iloc[idx, 1],
            c=config["fill"], edgecolors=config["edge"],
            s=70, alpha=0.9, linewidth=1.0, zorder=3
        )

# 6) Eixos centrais
plt.plot([0, 0], [-3, 3], "k-", zorder=1, alpha=0.5)
plt.plot([-5, 5], [0, 0], "k-", zorder=1, alpha=0.5)

# 7) Nomenclatura Científica dos Eixos (Obrigatório em Journals)
try:
    var_pc1 = modelo.results['explained_var'][0] * 100
    var_pc2 = modelo.results['explained_var'][1] * 100
    ax.set_xlabel(f"Principal Component 1 ({var_pc1:.1f}%)", fontweight='bold', fontsize=11)
    ax.set_ylabel(f"Principal Component 2 ({var_pc2:.1f}%)", fontweight='bold', fontsize=11)
except KeyError:
    # Fallback caso a chave seja diferente na sua versão da biblioteca
    ax.set_xlabel("Principal Component 1", fontweight='bold')
    ax.set_ylabel("Principal Component 2", fontweight='bold')

# 8) Legenda Rica
legend_elements = [
    Line2D([0], [0], marker='o', color='w', 
           label=f'C{k}: {v["name"]}', # Adiciona o nome geográfico
           markerfacecolor=v["fill"], markeredgecolor=v["edge"], markersize=10)
    for k, v in cluster_style.items() if k in set(clusters)
]
ax.legend(handles=legend_elements, title="Spatial Clusters", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)

plt.title("PCA Biplot of Wildfire Determinants in São Paulo State", pad=15, fontweight='bold')
plt.tight_layout()

# =========================
# 💾 SALVAR BIPLOT ATUAL
# =========================
save_path = "biplot_teste4.png"

fig.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Imagem salva com sucesso: {save_path}")

plt.show()

In [ ]:
# =========================
# ✅ BIPLOT PADRÃO PUBLICAÇÃO (Com Painel de Controle Total)
# =========================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import re

# ==========================================
# ⚙️ PAINEL DE CONTROLE (Edite à vontade)
# ==========================================
# 1. Setas e Textos das Variáveis
N_SETAS = 18
COR_SETA = "#444444"         # Ex: "black", "darkred", "#444444"
ESPESSURA_SETA = 10
COR_TEXTO_VAR = "black"
TAMANHO_TEXTO_VAR = 25
MOSTRAR_NUMEROS_VARIAVEIS = False # Mude para True se quiser ver os valores (ex: FF (0.24))
CASAS_DECIMAIS = 2           # Quantidade de casas decimais após a sigla

# 2. Tamanhos (Eixos, Escalas e Título)
TITULO_GRAFICO = "s " # <--- TÍTULO AQUI
TAMANHO_TITULO = 14
TAMANHO_NOME_EIXOS = 25
TAMANHO_NUMEROS_ESCALA = 20  # Números dos eixos X e Y

# 3. Legenda
TAMANHO_TEXTO_LEGENDA = 20
# Ordem exata que os clusters devem aparecer na legenda:
ORDEM_LEGENDA = [7, 5, 6, 4, 3, 1, 2, 0, 8, 9] 
# ==========================================


# 1) Carregar os clusters
df_clusters = pd.read_excel(ARQUIVO_CLUSTERS)
df_clusters.columns = df_clusters.columns.str.strip().str.lower()
clusters = df_clusters['cluster'].values

# 2) Nomes e Siglas
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

if "loadings" in modelo.results:
    modelo.results["loadings"] = modelo.results["loadings"].rename(columns=siglas_map)
if "topfeat" in modelo.results and isinstance(modelo.results["topfeat"], pd.DataFrame):
    if "feature" in modelo.results["topfeat"].columns:
        modelo.results["topfeat"]["feature"] = modelo.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

# 3) Dicionário de Cores e Nomes
cluster_style = {
    0: {"fill": "#f5d899", "edge": "#e69f00", "name": "Centro-Leste"}, 
    1: {"fill": "#bbe1f6", "edge": "#56b4e9", "name": "Nordeste"},
    2: {"fill": "#99d8c7", "edge": "#009e73", "name": "Litoral Sul"}, 
    3: {"fill": "#f9f4b3", "edge": "#f0e442", "name": "Norte"},
    4: {"fill": "#99c7e0", "edge": "#0072b2", "name": "Sudoeste"}, 
    5: {"fill": "#eebf99", "edge": "#d55e00", "name": "Leste"},
    6: {"fill": "#ebc9dc", "edge": "#cc79a7", "name": "Oeste"}, 
    7: {"fill": "#bbac9b", "edge": "#543005", "name": "Sul"},
    8: {"fill": "#99b6b6", "edge": "#004949", "name": "Centro-Oeste"}, 
    9: {"fill": "#d6d6d6", "edge": "#030303", "name": "Capital"}
}

# 4) Chama o biplot nativo injetando cor e espessura nas setas
modelo.biplot(
    n_feat=N_SETAS, 
    legend=False,
    arrowdict={'color': COR_SETA, 'linewidth': ESPESSURA_SETA, 'alpha': 0.8}
)

fig = plt.gcf()
ax = plt.gca()

# 🛠️ TRUQUE MÁGICO: Arredonda as casas decimais e aplica cor/tamanho nos textos
for txt in ax.texts:
    texto_atual = txt.get_text()
    if MOSTRAR_NUMEROS_VARIAVEIS:
        novo_texto = re.sub(r'\(([-0-9.]+)\)', lambda m: f"({float(m.group(1)):.{CASAS_DECIMAIS}f})", texto_atual)
    else:
        # Remove completamente qualquer coisa entre parênteses (ex: "FF (-0.23)" vira "FF")
        novo_texto = re.sub(r'\s*\([-0-9.]+\)', '', texto_atual)
        
    txt.set_text(novo_texto)
    txt.set_color(COR_TEXTO_VAR)
    txt.set_fontsize(TAMANHO_TEXTO_VAR)

# Oculta os pontos padrão
for collection in ax.collections:
    collection.set_visible(False)

# 5) Adiciona os pontos dos municípios
pcs = modelo.results['PC']  
for cluster_id, config in cluster_style.items():
    idx = (clusters == cluster_id)
    if idx.sum() > 0:
        ax.scatter(
            pcs.iloc[idx, 0], pcs.iloc[idx, 1],
            c=config["fill"], edgecolors=config["edge"],
            s=70, alpha=0.9, linewidth=1.0, zorder=3
        )

# 6) Eixos centrais
plt.plot([0, 0], [-3, 3], "k-", zorder=1, alpha=0.5)
plt.plot([-5, 5], [0, 0], "k-", zorder=1, alpha=0.5)

# 7) Estilização de Escalas, Títulos e Eixos
ax.tick_params(axis='both', which='major', labelsize=TAMANHO_NUMEROS_ESCALA)

try:
    # 🛠️ CORREÇÃO: 'variance_ratio' traz a variância individual (41.9 e 13.5)
    var_pc1 = modelo.results['variance_ratio'][0] * 100
    var_pc2 = modelo.results['variance_ratio'][1] * 100
    ax.set_xlabel(f"Principal Component 1 ({var_pc1:.1f}%)", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
    ax.set_ylabel(f"Principal Component 2 ({var_pc2:.1f}%)", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
except KeyError:
    ax.set_xlabel("Principal Component 1", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
    ax.set_ylabel("Principal Component 2", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)

# 8) Legenda Rica na ORDEM ESPECÍFICA
legend_elements = []
for k in ORDEM_LEGENDA:
    if k in set(clusters): # Garante que o cluster existe nos dados
        v = cluster_style[k]
        legend_elements.append(
            Line2D([0], [0], marker='o', color='w', 
                   label=f'{v["name"]}', 
                   markerfacecolor=v["fill"], markeredgecolor=v["edge"], markersize=10)
        )

ax.legend(
    handles=legend_elements, 
    title="Clusters", 
    title_fontsize=TAMANHO_TEXTO_LEGENDA+1,
    fontsize=TAMANHO_TEXTO_LEGENDA, 
    loc='upper right',       # <--- Coloca no canto superior direito
    frameon=True,            # <--- Ativa a caixa da legenda
    facecolor='white',       # <--- Fundo branco
    framealpha=0.6           # <--- Deixa 20% transparente para não esconder totalmente os pontos
)

plt.tight_layout()

# =========================
# 💾 SALVAR BIPLOT ATUAL
# =========================
save_path = "biplot_teste5.png"

fig.savefig(save_path, dpi=600, bbox_inches="tight", facecolor="white")
print(f"Imagem salva com sucesso: {save_path}")

plt.show()

In [ ]:
# =========================
# ✅ BIPLOT PADRÃO PUBLICAÇÃO (Sem título e limpo)
# =========================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import re

# ==========================================
# ⚙️ PAINEL DE CONTROLE (Edite à vontade)
# ==========================================
# 1. Setas e Textos das Variáveis
N_SETAS = 18
COR_SETA = "#444444"         # Ex: "black", "darkred", "#444444"
ESPESSURA_SETA = 10
COR_TEXTO_VAR = "black"
TAMANHO_TEXTO_VAR = 25
MOSTRAR_NUMEROS_VARIAVEIS = False # Mude para True se quiser ver os valores (ex: FF (0.24))
CASAS_DECIMAIS = 2           # Quantidade de casas decimais após a sigla

# 2. Tamanhos e Bordas
TAMANHO_NOME_EIXOS = 25
TAMANHO_NUMEROS_ESCALA = 25  # Números dos eixos X e Y
COR_BORDA = "black"          # Cor da borda externa do gráfico
ESPESSURA_BORDA = 1.5        # Espessura da borda externa

# 3. Legenda
TAMANHO_TEXTO_LEGENDA = 25
# Ordem exata que os clusters devem aparecer na legenda:
ORDEM_LEGENDA = [7, 5, 6, 4, 3, 1, 2, 0, 8, 9] 
# ==========================================

# 1) Carregar os clusters
df_clusters = pd.read_excel(ARQUIVO_CLUSTERS)
df_clusters.columns = df_clusters.columns.str.strip().str.lower()
clusters = df_clusters['cluster'].values

# 2) Nomes e Siglas
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

if "loadings" in modelo.results:
    modelo.results["loadings"] = modelo.results["loadings"].rename(columns=siglas_map)
if "topfeat" in modelo.results and isinstance(modelo.results["topfeat"], pd.DataFrame):
    if "feature" in modelo.results["topfeat"].columns:
        modelo.results["topfeat"]["feature"] = modelo.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

# 3) Dicionário de Cores e Nomes
cluster_style = {
    0: {"fill": "#f5d899", "edge": "#e69f00", "name": "Centro-Leste"}, 
    1: {"fill": "#bbe1f6", "edge": "#56b4e9", "name": "Nordeste"},
    2: {"fill": "#99d8c7", "edge": "#009e73", "name": "Litoral Sul"}, 
    3: {"fill": "#f9f4b3", "edge": "#f0e442", "name": "Norte"},
    4: {"fill": "#99c7e0", "edge": "#0072b2", "name": "Sudoeste"}, 
    5: {"fill": "#eebf99", "edge": "#d55e00", "name": "Leste"},
    6: {"fill": "#ebc9dc", "edge": "#cc79a7", "name": "Oeste"}, 
    7: {"fill": "#bbac9b", "edge": "#543005", "name": "Sul"},
    8: {"fill": "#99b6b6", "edge": "#004949", "name": "Centro-Oeste"}, 
    9: {"fill": "#d6d6d6", "edge": "#030303", "name": "Capital"}
}

# 4) Chama o biplot nativo injetando cor e espessura nas setas
modelo.biplot(
    n_feat=N_SETAS, 
    legend=False,
    title="", # <--- Bloqueia o título nativo da biblioteca
    arrowdict={'color': COR_SETA, 'linewidth': ESPESSURA_SETA, 'alpha': 0.8}
)

fig = plt.gcf()
ax = plt.gca()

# 🛠️ GOLPE DE MISERICÓRDIA NO TÍTULO: Garante que o topo fique totalmente em branco
ax.set_title("") 

# 🛠️ TRUQUE MÁGICO: Arredonda as casas decimais e aplica cor/tamanho nos textos
for txt in ax.texts:
    texto_atual = txt.get_text()
    if MOSTRAR_NUMEROS_VARIAVEIS:
        novo_texto = re.sub(r'\(([-0-9.]+)\)', lambda m: f"({float(m.group(1)):.{CASAS_DECIMAIS}f})", texto_atual)
    else:
        # Remove completamente qualquer coisa entre parênteses
        novo_texto = re.sub(r'\s*\([-0-9.]+\)', '', texto_atual)
        
    txt.set_text(novo_texto)
    txt.set_color(COR_TEXTO_VAR)
    txt.set_fontsize(TAMANHO_TEXTO_VAR)

# Oculta os pontos padrão
for collection in ax.collections:
    collection.set_visible(False)

# 5) Adiciona os pontos dos municípios
pcs = modelo.results['PC']  
for cluster_id, config in cluster_style.items():
    idx = (clusters == cluster_id)
    if idx.sum() > 0:
        ax.scatter(
            pcs.iloc[idx, 0], pcs.iloc[idx, 1],
            c=config["fill"], edgecolors=config["edge"],
            s=70, alpha=0.9, linewidth=1.0, zorder=3
        )

# 6) Eixos centrais
plt.plot([0, 0], [-3, 3], "k-", zorder=1, alpha=0.5)
plt.plot([-5, 5], [0, 0], "k-", zorder=1, alpha=0.5)

# 7) Estilização de Escalas, Eixos e BORDAS
ax.tick_params(axis='both', which='major', labelsize=TAMANHO_NUMEROS_ESCALA)

# Adiciona a borda preta nos 4 lados
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color(COR_BORDA)
    spine.set_linewidth(ESPESSURA_BORDA)

try:
    # 'variance_ratio' traz a variância individual (41.9 e 13.5)
    var_pc1 = modelo.results['variance_ratio'][0] * 100
    var_pc2 = modelo.results['variance_ratio'][1] * 100
    ax.set_xlabel(f"PC1 ({var_pc1:.1f}%)", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
    ax.set_ylabel(f"PC2 ({var_pc2:.1f}%)", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
except KeyError:
    ax.set_xlabel("PC1", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
    ax.set_ylabel("PC2", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)

# 8) Legenda Rica na ORDEM ESPECÍFICA
legend_elements = []
for k in ORDEM_LEGENDA:
    if k in set(clusters): 
        v = cluster_style[k]
        legend_elements.append(
            Line2D([0], [0], marker='o', color='w', 
                   label=f'{v["name"]}', 
                   markerfacecolor=v["fill"], markeredgecolor=v["edge"], markersize=10)
        )

ax.legend(
    handles=legend_elements, 
    title_fontsize=TAMANHO_TEXTO_LEGENDA+1,
    fontsize=TAMANHO_TEXTO_LEGENDA, 
    loc='upper right',       
    frameon=True,            
    facecolor='white',       
    framealpha=0.6           
)

plt.tight_layout()

# =========================
# 💾 SALVAR BIPLOT ATUAL
# =========================
save_path = "biplot_teste6.png"

fig.savefig(save_path, dpi=600, bbox_inches="tight", facecolor="white")
print(f"Imagem salva com sucesso: {save_path}")

plt.show()

## 7. PCA e biplot por cluster

In [ ]:
# ==========================================
# ✅ PCA E BIPLOT EXCLUSIVO POR CLUSTER (Apenas Siglas e com Borda)
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import re
import os
from pca import pca

# ==========================================
# ⚙️ PAINEL DE CONTROLE
# ==========================================
N_SETAS = 18
COR_SETA = "#444444"
ESPESSURA_SETA = 10
COR_TEXTO_VAR = "black"
TAMANHO_TEXTO_VAR = 25

# 👇 MANTENHA FALSE PARA ESCONDER OS NÚMEROS
MOSTRAR_NUMEROS_VARIAVEIS = False 
CASAS_DECIMAIS = 2

# Tamanhos e Bordas
TAMANHO_NOME_EIXOS = 25
TAMANHO_NUMEROS_ESCALA = 25
TAMANHO_TEXTO_LEGENDA = 25
COR_BORDA = "black"          # Cor da borda externa do gráfico
ESPESSURA_BORDA = 1.5        # Espessura da borda externa

DIR_SALVAR = "/PCA_geral_e_clusters/"
# ==========================================

# 1) Carregar os clusters
path_clusters = os.path.join(DIR_SALVAR, "BANCO_SEPARADO_POR_CLUSTER.xlsx")
df_clusters = pd.read_excel(path_clusters)
df_clusters.columns = df_clusters.columns.str.strip().str.lower()
clusters = df_clusters['cluster'].values

# 2) Dicionário de Cores e Nomes
cluster_style = {
    0: {"fill": "#f5d899", "edge": "#e69f00", "name": "Centro-Leste"}, 
    1: {"fill": "#bbe1f6", "edge": "#56b4e9", "name": "Nordeste"},
    2: {"fill": "#99d8c7", "edge": "#009e73", "name": "Litoral Sul"}, 
    3: {"fill": "#f9f4b3", "edge": "#f0e442", "name": "Norte"},
    4: {"fill": "#99c7e0", "edge": "#0072b2", "name": "Sudoeste"}, 
    5: {"fill": "#eebf99", "edge": "#d55e00", "name": "Leste"},
    6: {"fill": "#ebc9dc", "edge": "#cc79a7", "name": "Oeste"}, 
    7: {"fill": "#bbac9b", "edge": "#543005", "name": "Sul"},
    8: {"fill": "#99b6b6", "edge": "#004949", "name": "Centro-Oeste"}, 
    9: {"fill": "#d6d6d6", "edge": "#030303", "name": "Capital"}
}

# 3) Mapeamento de Siglas
vars_orig = list(dados_pad_cleaned.columns)
siglas_map = {v: siglas.get(v, v) for v in vars_orig}

# ==========================================
# 🔄 LOOP: NOVO PCA E BIPLOT POR CLUSTER
# ==========================================
for cluster_id, config in cluster_style.items():
    
    idx = (clusters == cluster_id)
    
    if idx.sum() < 3:
        continue
        
    # 4) FILTRA OS DADOS E RODA UM NOVO PCA
    dados_cluster = dados_pad_cleaned[idx]
    
    modelo_cluster = pca(n_components=2, normalize=False) 
    modelo_cluster.fit_transform(dados_cluster)
    
    # Aplica as siglas no novo modelo
    if "loadings" in modelo_cluster.results:
        modelo_cluster.results["loadings"] = modelo_cluster.results["loadings"].rename(columns=siglas_map)
    if "topfeat" in modelo_cluster.results and isinstance(modelo_cluster.results["topfeat"], pd.DataFrame):
        if "feature" in modelo_cluster.results["topfeat"].columns:
            modelo_cluster.results["topfeat"]["feature"] = modelo_cluster.results["topfeat"]["feature"].map(lambda x: siglas_map.get(x, x))

    # 5) Gera o Biplot nativo do novo modelo
    modelo_cluster.biplot(
        n_feat=N_SETAS, 
        legend=False,
        title="", 
        arrowdict={'color': COR_SETA, 'linewidth': ESPESSURA_SETA, 'alpha': 0.8}
    )

    fig = plt.gcf()
    ax = plt.gca()
    ax.set_title("") 

    # 6) AJUSTE DE TEXTOS: Remove tudo entre parênteses se MOSTRAR_NUMEROS_VARIAVEIS for False
    for txt in ax.texts:
        texto_atual = txt.get_text()
        if MOSTRAR_NUMEROS_VARIAVEIS:
            novo_texto = re.sub(r'\(([-0-9.]+)\)', lambda m: f"({float(m.group(1)):.{CASAS_DECIMAIS}f})", texto_atual)
        else:
            # Limpeza total: remove qualquer parêntese e seu conteúdo
            novo_texto = re.sub(r'\s*\(.*?\)', '', texto_atual)
            
        txt.set_text(novo_texto)
        txt.set_color(COR_TEXTO_VAR)
        txt.set_fontsize(TAMANHO_TEXTO_VAR)

    # Oculta os pontos padrão do biplot
    for collection in ax.collections:
        collection.set_visible(False)

    # 7) Plota os pontos exclusivos deste cluster
    pcs_cluster = modelo_cluster.results['PC']
    ax.scatter(
        pcs_cluster.iloc[:, 0], pcs_cluster.iloc[:, 1],
        c=config["fill"], edgecolors=config["edge"],
        s=70, alpha=0.9, linewidth=1.0, zorder=3
    )

    # Linhas centrais
    plt.plot([0, 0], [-3, 3], "k-", zorder=1, alpha=0.5)
    plt.plot([-5, 5], [0, 0], "k-", zorder=1, alpha=0.5)
    
    # Estilização das escalas
    ax.tick_params(axis='both', which='major', labelsize=TAMANHO_NUMEROS_ESCALA)

    # 8) ADICIONA A BORDA PRETA NOS 4 LADOS (Novo!)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(COR_BORDA)
        spine.set_linewidth(ESPESSURA_BORDA)

    # 9) Percentuais de variância específicos
    try:
        var_pc1 = modelo_cluster.results['variance_ratio'][0] * 100
        var_pc2 = modelo_cluster.results['variance_ratio'][1] * 100
        ax.set_xlabel(f"PC1 ({var_pc1:.1f}%)", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
        ax.set_ylabel(f"PC2 ({var_pc2:.1f}%)", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
    except KeyError:
        ax.set_xlabel("PC1", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)
        ax.set_ylabel("PC2", fontweight='bold', fontsize=TAMANHO_NOME_EIXOS)

    # 10) Legenda do Cluster
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', 
               label=f'{config["name"]}', 
               markerfacecolor=config["fill"], markeredgecolor=config["edge"], markersize=10)
    ]

    ax.legend(
        handles=legend_elements, 
        title_fontsize=TAMANHO_TEXTO_LEGENDA+1,
        fontsize=TAMANHO_TEXTO_LEGENDA, 
        loc='upper right',       
        frameon=True,            
        facecolor='white',       
        framealpha=0.6           
    )

    plt.tight_layout()

    # 11) Salvar imagem
    nome_arquivo = f"Novo_PCA_Cluster_{cluster_id}_{config['name'].replace(' ', '_')}.png"
    save_path = os.path.join(DIR_SALVAR, nome_arquivo)
    
    fig.savefig(save_path, dpi=600, bbox_inches="tight", facecolor="white")
    print(f"✅ Salvo: {save_path}")

    plt.close(fig)

print("\n🚀 Concluído! Gráficos gerados com as bordas ajustadas.")